In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Extraer la informacion del archivo.
import pandas as pd

vaccinations = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/country_vaccinations.csv')

In [ ]:
# Mostrar la estructura y tipos de datos de cada columna para identificar que
# operaciones puedes realizar con cada una de ellas

vaccinations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86512 entries, 0 to 86511
Data columns (total 15 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   country                              86512 non-null  object 
 1   iso_code                             86512 non-null  object 
 2   date                                 86512 non-null  object 
 3   total_vaccinations                   43607 non-null  float64
 4   people_vaccinated                    41294 non-null  float64
 5   people_fully_vaccinated              38802 non-null  float64
 6   daily_vaccinations_raw               35362 non-null  float64
 7   daily_vaccinations                   86213 non-null  float64
 8   total_vaccinations_per_hundred       43607 non-null  float64
 9   people_vaccinated_per_hundred        41294 non-null  float64
 10  people_fully_vaccinated_per_hundred  38802 non-null  float64
 11  daily_vaccinations_per_milli

In [ ]:
#cambiando las columnas con fechas para que sean del tipo datetime64

vaccinations['date'] = pd.to_datetime(vaccinations['date'])
vaccinations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86512 entries, 0 to 86511
Data columns (total 15 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   country                              86512 non-null  object        
 1   iso_code                             86512 non-null  object        
 2   date                                 86512 non-null  datetime64[ns]
 3   total_vaccinations                   43607 non-null  float64       
 4   people_vaccinated                    41294 non-null  float64       
 5   people_fully_vaccinated              38802 non-null  float64       
 6   daily_vaccinations_raw               35362 non-null  float64       
 7   daily_vaccinations                   86213 non-null  float64       
 8   total_vaccinations_per_hundred       43607 non-null  float64       
 9   people_vaccinated_per_hundred        41294 non-null  float64       
 10  people_ful

In [ ]:
# Determinar la cantidad de vacunas aplicadas de cada compania (con base en como
# lo reporta cada pais en la columna vaccines, en otras palabras, agrupe por
# vaccines y realice la sumatoria)

#Aseguramos que total_vaccinations sea numerico y sin NaN
vaccinations['total_vaccinations'] = pd.to_numeric(vaccinations['total_vaccinations'], errors='coerce').fillna(0)

#Nos quedamos con el ultimo registro de total_vaccinations de cada pais
latest_vaccination = vaccinations.sort_values('date').groupby('country').last().reset_index()

#Separamos las vacunas en una lista
def safe_split(v):
    if isinstance(v, str):
        return [x.strip() for x in v.split(',')]
    else:
        return []  # si es NaN o float, devolvemos lista vacía

latest_vaccination['vaccines'] = latest_vaccination['vaccines'].apply(safe_split)

#Usamos la funcion explode para que cada vacuna tenga su propia fila
latest_vaccination_exploded = latest_vaccination.explode('vaccines')

#Agrupamos por vacuna y sumamos
vaccines_by_country = latest_vaccination_exploded.groupby('vaccines', as_index=False)['total_vaccinations'].sum()
vaccines_by_country = vaccines_by_country.sort_values('total_vaccinations', ascending=False)
#Muestra el resultado
print(vaccines_by_country)

              vaccines  total_vaccinations
10  Oxford/AstraZeneca        6.947569e+09
16             Sinovac        5.832917e+09
14   Sinopharm/Beijing        5.666422e+09
11     Pfizer/BioNTech        5.551421e+09
8              Moderna        4.291180e+09
2              CanSino        3.923173e+09
6      Johnson&Johnson        3.559488e+09
21           Sputnik V        3.536245e+09
23              ZF2001        3.314887e+09
15     Sinopharm/Wuhan        3.287593e+09
3              Covaxin        2.242632e+09
9              Novavax        8.925558e+08
0               Abdala        2.859597e+08
20       Sputnik Light        2.609835e+08
18          Soberana02        2.296079e+08
4         EpiVacCorona        1.711821e+08
22            Turkovac        1.468819e+08
19            SpikoGen        1.467926e+08
1      COVIran Barekat        1.467926e+08
5            FAKHRAVAC        1.467926e+08
13       Razi Cov Pars        1.467926e+08
7              Medigen        4.948130e+07
17       So

In [ ]:
#Obtener la cantidad de vacunas aplicadas en todo el mundo

total_vaccinations_world = latest_vaccination['total_vaccinations'].sum()
print(f"La cantidad de vacunas aplicadas en todo el mundo es: {total_vaccinations_world}")


La cantidad de vacunas aplicadas en todo el mundo es: 11308135686.0


In [ ]:
#Calcular el promedio de vacunas aplicadas por pais

average_by_country = vaccinations.groupby('country', as_index = False)['daily_vaccinations'].mean().rename(columns={'daily_vaccinations':'average_by_country'})
print(average_by_country)

               country  average_by_country
0          Afghanistan        14610.681934
1              Albania         6276.210046
2              Algeria        33904.356436
3              Andorra          367.716019
4               Angola        44821.457584
..                 ...                 ...
218              Wales        15518.411765
219  Wallis and Futuna           33.886486
220              Yemen         2556.115756
221             Zambia         9649.805158
222           Zimbabwe        21669.066832

[223 rows x 2 columns]


In [ ]:
#Determinar la cantidad de vacunas aplicadas del dia 29/01/21 en todo el
#mundo

#filramos por la fecha
date = '29/01/21'
vaccines_day = vaccinations[vaccinations['date'] == date]

#sumar la cantidad
total_vaccinations_day = vaccines_day['daily_vaccinations'].sum()

print('Cantidad total de vacunas aplicadas en el mundo el 29/01/21:',total_vaccinations_day)



Cantidad total de vacunas aplicadas en el mundo el 29/01/21: 4884052.0


In [ ]:
# Crear un dataframe nuevo denominado conDiferencias que contenga los datos
# originales y una columna derivada (diferencias) con las diferencias de
# aplicacion entre las columnas daily_vaccinations y daily_vaccinations_raw

#Nos aseguramos de que ambas columnas sean numericas (por si hay valores vacios)
vaccinations['daily_vaccinations'] = pd.to_numeric(vaccinations['daily_vaccinations'], errors='coerce')
vaccinations['daily_vaccinations_raw'] = pd.to_numeric(vaccinations['daily_vaccinations_raw'], errors='coerce')

conDiferencias = vaccinations.copy()
conDiferencias['diferencias'] = conDiferencias['daily_vaccinations'] - conDiferencias['daily_vaccinations_raw']


In [ ]:
# Obtener el periodo de tiempo entre el registro con fecha mas reciente y el
# registro con fecha mas antigua

vaccinations['date'] = pd.to_datetime(vaccinations['date'])

min_date = vaccinations['date'].min()
max_date = vaccinations['date'].max()

period = max_date - min_date


print('Fecha mas antigua:', min_date.date())
print('Fecha mas reciente:', max_date.date())
print('Periodo de tiempo:', period.days)

Fecha mas antigua: 2020-12-02
Fecha mas reciente: 2022-03-29
Periodo de tiempo: 482


In [ ]:
# Crear un dataframe nuevo denominado conCantidad que contenga los datos
# originales y una columna derivada (canVac) con la cantidad de vacunas
# utilizadas cada dia (usar la columna vaccines y separar por el caracter , )

conCantidad = vaccinations.copy()

#crear la columna con la cantidad de vacunas por fila
conCantidad['canVac'] = conCantidad['vaccines'].str.split(',').str.len()


In [ ]:
# Generar un dataframe denominado antes20 con todos los registros que se
#hayan realizado antes del 20 de diciembre de 2020

antes20 = vaccinations.copy()
antes20 = vaccinations[vaccinations['date'] < '20/12/21']


In [ ]:
# Obtener un dataframe denominado pfizer con todos los registros donde se haya
# utilizado la vacuna Pfizer

pfizer = vaccinations.copy()
pfizer = vaccinations[vaccinations['vaccines'].str.contains('Pfizer', case = False, na = False)]


In [ ]:
# Almacenar los dataframes generados (conDiferencias, conCantidad, antes20 y
# pfizer) en un archivo de Excel denominado resultadosReto.xlsx, donde cada
# dataframe ocupe una hoja diferente.

!pip install xlsxwriter

ruta = '/content/drive/MyDrive/ColabNotebooks/resultadosReto.xlsx'

with pd.ExcelWriter(ruta, engine = 'xlsxwriter')as writer:
  conDiferencias.to_excel(writer, sheet_name = 'Diferencias', index = False)
  conCantidad.to_excel(writer, sheet_name = 'Cantidad', index = False)
  antes20.to_excel(writer, sheet_name = 'Antes20', index = False)
  pfizer.to_excel(writer, sheet_name = 'Pfizer', index = False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 3.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/xlsxwriter/worksheet.py:1321: UserWarning: Ignoring URL 'https://ais.paho.org/imm/IM_DosisAdmin-Vacunacion.asp' since it exceeds Excel's limit of 65,530 URLs per worksheet.
  warn(
/usr/local/lib/python3.12/dist-packages/xlsxwriter/worksheet.py:1321: UserWarning: Ignoring URL 'https://web.facebook.com/SVGHEALTH/posts/437021804887233' since it exceeds Excel's limit of 65,530 URLs per worksheet.
  warn(
/usr/local/lib/python3.12/dist-packages/xlsxwriter/worksheet.py:1321: UserWarning: Ignoring URL 'https://stats.pacificdata.org/vis?tm=covid&pg=0&df[ds]=SPC2&df[id]=DF_COVID_VACCINATION&df[ag]=SPC&df[vs]=1.0' since it exceeds Excel's limit of 65,530 URLs per worksheet.
  warn(
/usr/local/lib/python3.12/dist-packages/xlsxwriter/worksheet.py:1321: UserWarning: Ignoring URL 'https://covid19.who.int/' since it exceeds Excel's limit of 65,530 URLs per worksheet.
  warn(
/usr/local/lib/python3.12/dist-packages/xlsxwriter/worksheet.py:1321: UserWarning: Ign